## Demo — Reading a Sub-policy Demographic-Parity Ratio (SafeWheels Insurance)

**Scenario:** SafeWheels' auto-insurance pricing model returns *Eligible for preferred-rate tier* (1 / 0). The audit team has policy thresholds for the three core fairness metrics (DP ratio ≥ 0.80, equalized-odds difference ≤ 0.10, equal-opportunity difference ≤ 0.10).

This walkthrough computes the three metrics **from confusion-matrix primitives first**, then verifies them against **Fairlearn** as a one-line cross-check. We interpret the numbers, run a counterfactual on per-group threshold adjustment, and identify the launch decision.

> **Further reading.** Cross-method validation: **AIF360**, **Aequitas**. LLM-fairness benchmarks: **BBQ**, **BiasInBios**, **HELM**.

In [1]:
import pandas as pd
import numpy as np
from fairlearn.metrics import (
    MetricFrame, demographic_parity_ratio, equalized_odds_difference, true_positive_rate
)
from sklearn.metrics import accuracy_score

df = pd.read_csv("data/insurance_predictions.csv")
df.head()

,age_band,region,label,prediction
0,Under 25,Urban,0,0
1,Under 25,Urban,1,0
2,Under 25,Urban,0,0
3,Under 25,Urban,0,0
4,Under 25,Urban,1,0


## 1. Compute the three core metrics from confusion-matrix primitives

In [2]:
def selection_rate(y_pred):
    return float((y_pred == 1).mean())

def tpr(y_true, y_pred):
    pos = (y_true == 1)
    return float((y_pred[pos] == 1).mean()) if pos.sum() else float('nan')

def fpr(y_true, y_pred):
    neg = (y_true == 0)
    return float((y_pred[neg] == 1).mean()) if neg.sum() else float('nan')

def dp_ratio_from_primitives(y_true, y_pred, group):
    rates = {g: selection_rate(y_pred[group == g]) for g in group.unique()}
    return min(rates.values()) / max(rates.values())

def eo_diff_from_primitives(y_true, y_pred, group):
    # max gap across groups in either TPR or FPR
    tprs = {g: tpr(y_true[group == g], y_pred[group == g]) for g in group.unique()}
    fprs = {g: fpr(y_true[group == g], y_pred[group == g]) for g in group.unique()}
    return max(max(tprs.values()) - min(tprs.values()),
               max(fprs.values()) - min(fprs.values()))

def eopp_diff_from_primitives(y_true, y_pred, group):
    tprs = {g: tpr(y_true[group == g], y_pred[group == g]) for g in group.unique()}
    return max(tprs.values()) - min(tprs.values())

y_true = df['label']
y_pred = df['prediction']
g      = df['age_band']

primitives = {
    'dp_ratio':  dp_ratio_from_primitives(y_true, y_pred, g),
    'eo_diff':   eo_diff_from_primitives(y_true, y_pred, g),
    'eopp_diff': eopp_diff_from_primitives(y_true, y_pred, g),
    'accuracy':  float((y_true == y_pred).mean()),
}
primitives

{'dp_ratio': 0.7292732855680656,
 'eo_diff': 0.270846362309777,
 'eopp_diff': 0.270846362309777,
 'accuracy': 0.7712903225806451}

### 1b. Fairlearn cross-check (one line)

Sanity-check the primitives against the library's reference implementations.

In [3]:
from fairlearn.metrics import (
    demographic_parity_ratio, equalized_odds_difference, MetricFrame, true_positive_rate
)
from sklearn.metrics import accuracy_score

fl_dp     = demographic_parity_ratio(y_true=y_true, y_pred=y_pred, sensitive_features=g)
fl_eo     = equalized_odds_difference(y_true=y_true, y_pred=y_pred, sensitive_features=g)
fl_tpr_mf = MetricFrame(metrics=true_positive_rate, y_true=y_true, y_pred=y_pred,
                       sensitive_features=g)
fl_eopp   = float(fl_tpr_mf.by_group.max() - fl_tpr_mf.by_group.min())
fl_acc    = float(accuracy_score(y_true, y_pred))

print(f'  primitives  →  DP={primitives["dp_ratio"]:.4f}  EO={primitives["eo_diff"]:.4f}  EOpp={primitives["eopp_diff"]:.4f}')
print(f'  fairlearn   →  DP={fl_dp:.4f}  EO={fl_eo:.4f}  EOpp={fl_eopp:.4f}')
assert abs(primitives['dp_ratio']  - fl_dp)   < 1e-9
assert abs(primitives['eo_diff']   - fl_eo)   < 1e-9
assert abs(primitives['eopp_diff'] - fl_eopp) < 1e-9
print('cross-check: primitives match Fairlearn within 1e-9 ✓')

  primitives  →  DP=0.7293  EO=0.2708  EOpp=0.2708
  fairlearn   →  DP=0.7293  EO=0.2708  EOpp=0.2708
cross-check: primitives match Fairlearn within 1e-9 ✓


## 2. Read the numbers

DP ratio is sub-policy (< 0.80) → **fails the demographic-parity floor**. Equalized-odds + equal-opportunity differences also breach their 0.10 ceilings. The system is technically launchable from an accuracy view, but multiple policy thresholds are binding.

## 3. Counterfactual — per-group threshold adjustment

The current model uses a single global threshold. We can ship per-group thresholds that move the worst-performing group up. We don't have raw scores in this demo (the prediction is already 0/1), so the counterfactual here is illustrative — see the exercise notebook for a real per-group threshold sweep using a `score` column.

## 4. Launch decision

**Recommendation: Conditional Launch.** The model can ship contingent on:
1. Per-group threshold adjustment for the under-25 cell to bring DP ratio above 0.80.
2. Monthly monitoring of the three core metrics with paging at any policy-threshold breach.
3. 90-day post-launch fairness re-review at the AI review board.

**Owner:** AI Risk Officer (audit), Pricing Model Owner (implementation), AI Review Board (sign-off).

**Key takeaway:** fairness metrics are inputs to a governance decision, not the decision itself. The interpretation layer — "what does the computed ratio mean for THIS product, in THIS regulatory context?" — is the GRC practitioner's job.

## Reference Notes

A few specification details for the fairness-policy thresholds and library APIs used above:

- **0.10 ceiling for equalized-odds and equal-opportunity differences.** The 0.10 threshold used in the SafeWheels / UdaciBank scenarios is a **firm-policy threshold** chosen for this lesson, not a regulatory or industry-wide standard. Real organizations set EO / EOpp ceilings based on their policy, regulatory exposure, and the distribution of their training data; common firm policies range from 0.05 to 0.15 depending on use case and risk tier. By contrast, the **0.80 demographic-parity floor** is anchored to the EEOC 4/5ths rule and has external regulatory provenance.
- **`ThresholdOptimizer` is post-processing per-group threshold optimization.** It selects a separate decision threshold per protected group to satisfy a fairness constraint (e.g., equalized odds). It does not modify the training data — that's the role of pre-processing techniques like AIF360's `Reweighing`. Both families are valid mitigations; they intervene at different points in the pipeline.
- **The fairness-impossibility result.** The formal impossibility result (Chouldechova 2017; Kleinberg–Mullainathan–Raghavan 2016) is between **calibration** and **error-rate balance under unequal base rates**. The "impossibility trilemma" framing across DP, EO, and EOpp is a related practical observation — these metrics generally cannot all hold simultaneously except in degenerate cases — but the formal proof targets the calibration-vs-equalized-odds pair specifically.